# News Article Topic Modeling

Discover recurring themes in an unlabeled news corpus and explain them with top terms.

**Portfolio category:** NLP and topic modelling

**Data mode:** Verified demo mode

This notebook keeps labels out of fitting wherever labels exist, uses deterministic seeds,
reports unsupervised-specific diagnostics, and avoids hard-coded results.

## 1. Project setup

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

## 2. Build an auditable demonstration corpus

In [ ]:
themes = {'artificial_intelligence': ['new model improves reasoning', 'researchers evaluate machine learning safety', 'companies adopt generative AI tools', 'chip demand grows for data centres'], 'finance': ['central bank discusses interest rates', 'markets react to inflation data', 'investors review quarterly earnings', 'currency volatility affects trade'], 'health': ['clinical study tests a new treatment', 'public health team tracks disease patterns', 'hospital adopts digital diagnostics', 'research links sleep and wellbeing'], 'climate': ['cities prepare for extreme heat', 'renewable energy capacity expands', 'scientists monitor ocean temperatures', 'policy targets industrial emissions'], 'sport': ['team wins after a late comeback', 'coach changes the starting lineup', 'tournament reaches the final round', 'athlete breaks a national record']}
documents = []
hidden_theme = []
for theme, phrases in themes.items():
    for _ in range(28):
        selected = rng.choice(phrases, size=3, replace=True)
        documents.append(". ".join(selected))
        hidden_theme.append(theme)
corpus = pd.DataFrame({"document": documents, "hidden_theme": hidden_theme}).sample(
    frac=1, random_state=RANDOM_STATE
).reset_index(drop=True)
display(corpus.head())

## 3. Corpus quality

In [ ]:
corpus["word_count"] = corpus["document"].str.split().str.len()
display(corpus["word_count"].describe().to_frame())
print("Duplicate documents:", int(corpus["document"].duplicated().sum()))

## 4. TF-IDF features

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=3, max_df=0.95)
X = vectorizer.fit_transform(corpus["document"])
print("Document-term matrix:", X.shape)

## 5. Fit NMF topics

In [ ]:
n_topics = 5
model = NMF(n_components=n_topics, init="nndsvda", random_state=RANDOM_STATE, max_iter=600)
document_topics = model.fit_transform(X)
terms = np.array(vectorizer.get_feature_names_out())
topic_terms = {}
for topic_id, weights in enumerate(model.components_):
    topic_terms[topic_id] = list(terms[np.argsort(weights)[-8:][::-1]])
display(pd.DataFrame.from_dict(topic_terms, orient="index"))
corpus["topic"] = document_topics.argmax(axis=1)
corpus["topic_strength"] = document_topics.max(axis=1)

## 6. Topic quality diagnostics

In [ ]:
flattened_terms = [term for words in topic_terms.values() for term in words]
topic_diversity = len(set(flattened_terms)) / len(flattened_terms)
concentration = corpus["topic_strength"].mean()
display(pd.Series({
    "reconstruction_error": model.reconstruction_err_,
    "topic_diversity": topic_diversity,
    "mean_topic_strength": concentration,
}).to_frame("value"))

## 7. Topic prevalence

In [ ]:
prevalence = corpus["topic"].value_counts(normalize=True).sort_index()
prevalence.plot.bar(color="#2563eb")
plt.title("Topic prevalence")
plt.ylabel("Share of documents")
plt.tight_layout()

## 8. Inspect representative documents

In [ ]:
representatives = corpus.sort_values("topic_strength", ascending=False).groupby("topic").head(3)
display(representatives[["topic", "topic_strength", "document", "hidden_theme"]])

## 9. Key findings

Topic labels should be assigned from top terms and representative documents, not from arbitrary topic numbers.

## 10. Interpretation and responsible use

Treat the output as exploratory evidence, not ground truth. For news article topic modeling,
validate stability on newer data, inspect edge cases, and review domain risks before
turning clusters, rankings or anomaly scores into decisions.

## 11. Next steps

- Replace demonstration data with a versioned, licensed dataset.
- Track data quality, drift and stability across repeated runs.
- Add domain-specific review before deployment.
- Package inference only after reproducibility and privacy checks pass.

All numeric results are generated at execution time; none are hard-coded.